In [1]:
import time
from google import genai
import dotenv

dotenv.load_dotenv()
client = genai.Client()

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = "batches/nf9cvhb3lj3dqip9c80dbk9bzo79wptqorpb"  # (e.g. 'batches/your-batch-id')
batch_job = client.batches.get(name=job_name)

completed_states = set([
    'JOB_STATE_SUCCEEDED',
    'JOB_STATE_FAILED',
    'JOB_STATE_CANCELLED',
    'JOB_STATE_EXPIRED',
])

print(f"Polling status for job: {job_name}")
batch_job = client.batches.get(name=job_name) # Initial get
while batch_job.state.name not in completed_states:
  print(f"Current state: {batch_job.state.name}")
  time.sleep(30) # Wait for 30 seconds before polling again
  batch_job = client.batches.get(name=job_name)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

Polling status for job: batches/nf9cvhb3lj3dqip9c80dbk9bzo79wptqorpb
Job finished with state: JOB_STATE_EXPIRED


In [2]:
import os
import sys
import json
from pathlib import Path

from dotenv import load_dotenv

# Repo root (assumes this notebook lives in project root)
ROOT = Path.cwd()

# Load API keys and other config
load_dotenv(ROOT / ".env")

# Make batch helper modules importable
scripts_dir = ROOT / "scripts"
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from scripts import batch_openai
from scripts import batch_gemini

DEFAULT_BATCH_IDS_PATHS = [
    ROOT / "data" / "batch_ids.json",
    ROOT / "batch_ids.json",
]


def load_batches(path: Path | None = None):
    """Return (path, batches) using batch_ids.json from data/ or root."""
    if path is None:
        for p in DEFAULT_BATCH_IDS_PATHS:
            if p.is_file():
                path = p
                break
    if path is None:
        raise FileNotFoundError("No batch_ids.json found in data/ or repo root")
    with path.open() as f:
        data = json.load(f)
    return path, data.get("batches", [])


print("Setup complete; helpers imported.")

Setup complete; helpers imported.


In [3]:
from pprint import pprint

BATCH_IDS_PATH, batches = load_batches()
terminal_statuses = {"completed", "failed", "cancelled", "expired"}

print(f"Loaded {len(batches)} batches from {BATCH_IDS_PATH}")
active = []

for b in batches:
    bid = b.get("batch_id")
    provider = b.get("provider")
    model = b.get("model")
    if not bid or provider not in ("openai", "gemini"):
        continue
    if provider == "openai":
        st = batch_openai.get_openai_batch_status(bid)
    else:
        st = batch_gemini.get_gemini_batch_status(bid)
    status = st.get("status")
    print(f"{provider}/{model}: {bid} -> {status}")
    if status not in terminal_statuses:
        active.append({**b, "status": status})

print("\nActive (non-terminal) batches:")
pprint(active)

len(active)

Loaded 16 batches from /Users/georgiiburdi/DataspellProjects/benchmark_eval/data/batch_ids.json
openai/gpt-5.2: batch_69a6e4af0c848190b77c7cfb10a7d4ba -> completed
openai/gpt-5.1: batch_69a6e4b20f888190ba3506693ace75e9 -> cancelling
openai/gpt-5: batch_69a6e4b566248190bc40335e12719465 -> cancelling
openai/gpt-5-mini: batch_69a6e4b8d8f88190a79988192c68fece -> cancelling
openai/gpt-5-nano: batch_69a6e4bbde8c81909f1acdd10a10491b -> completed
openai/gpt-4.1: batch_69a6e4bf41b08190948807f5a5bea027 -> completed
openai/gpt-4.1-mini: batch_69a6e4c26ef08190a8df03bd8caead25 -> cancelled
openai/gpt-4.1-nano: batch_69a6e4c5d5288190834299ecef71ede4 -> cancelled
openai/gpt-4o: batch_69a6e4c8ad048190a2d96a165c829426 -> cancelled
gemini/gemini-3-flash-preview: batches/b1uzn7oambm424uecm7nl1u1lkxqtkygr9tc -> cancelled
gemini/gemini-2.5-flash: batches/2z65mj43qnfjlpwww6fu3qqi65qadh5b9lbl -> cancelled
gemini/gemini-2.5-pro: batches/5i3gh2mah1amf8j1k0qh84fdkd1sanl1hfue -> cancelled
gemini/gemini-2.0-flash

3

In [4]:
# Cancel all active OpenAI + Gemini batches from the previous cell

if "active" not in globals():
    raise RuntimeError("Run the status cell first to populate 'active'.")

cancelled = []
for b in active:
    bid = b["batch_id"]
    provider = b["provider"]
    model = b["model"]
    if provider == "openai":
        st = batch_openai.cancel_openai_batch(bid)
    elif provider == "gemini":
        st = batch_gemini.cancel_gemini_batch(bid)
    else:
        continue
    print(f"Requested cancel: {provider}/{model} {bid} -> {st.get('status')}")
    cancelled.append({**b, "after_cancel": st})

cancelled

Requested cancel: openai/gpt-5.1 batch_69a6e4b20f888190ba3506693ace75e9 -> cancelling
Requested cancel: openai/gpt-5 batch_69a6e4b566248190bc40335e12719465 -> cancelling
Requested cancel: openai/gpt-5-mini batch_69a6e4b8d8f88190a79988192c68fece -> cancelling


[{'model': 'gpt-5.1',
  'provider': 'openai',
  'batch_id': 'batch_69a6e4b20f888190ba3506693ace75e9',
  'status': 'cancelling',
  'after_cancel': {'id': 'batch_69a6e4b20f888190ba3506693ace75e9',
   'status': 'cancelling',
   'output_file_id': None,
   'error_file_id': None}},
 {'model': 'gpt-5',
  'provider': 'openai',
  'batch_id': 'batch_69a6e4b566248190bc40335e12719465',
  'status': 'cancelling',
  'after_cancel': {'id': 'batch_69a6e4b566248190bc40335e12719465',
   'status': 'cancelling',
   'output_file_id': None,
   'error_file_id': None}},
 {'model': 'gpt-5-mini',
  'provider': 'openai',
  'batch_id': 'batch_69a6e4b8d8f88190a79988192c68fece',
  'status': 'cancelling',
  'after_cancel': {'id': 'batch_69a6e4b8d8f88190a79988192c68fece',
   'status': 'cancelling',
   'output_file_id': None,
   'error_file_id': None}}]

In [1]:
import os, requests, pandas as pd
from requests.auth import HTTPBasicAuth

host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
pub = os.environ["LANGFUSE_PUBLIC_KEY"]
sec = os.environ["LANGFUSE_SECRET_KEY"]

auth = HTTPBasicAuth(pub, sec)

params = {
    "limit": 100,
    "page": 1,
    "fromTimestamp": "2025-01-01T00:00:00Z",  # very old
}

r = requests.get(f"{host}/api/public/v2/scores", auth=auth, params=params)
r.raise_for_status()
data = r.json()["data"]

df = pd.DataFrame(data)
print("min timestamp:", df["timestamp"].min() if "timestamp" in df else None)
print("distinct names:", sorted(set(d.get("name") for d in data)))

min timestamp: 2026-03-04T17:28:43.358Z
distinct names: ['accuracy']


In [4]:
import os
import sys
import json
import csv
from pathlib import Path
import dotenv
dotenv.load_dotenv()
# --- setup imports -----------------------------------------------------------
ROOT = Path.cwd()  # assuming notebook is at repo root
scripts_lib = ROOT / "scripts" / "lib"
if str(scripts_lib) not in sys.path:
    sys.path.insert(0, str(scripts_lib))

from scripts.lib.load_dataset import load_dataset_rows  # uses DEFAULT_LANGFUSE_DATASET or LANGFUSE_DATASET_NAME

# --- config ------------------------------------------------------------------
BASELINE_PATH = ROOT / "data" / "baseline_predictions.json"

# Langfuse dataset that contains the *full* campaign relevance data
# If you set LANGFUSE_DATASET_NAME in .env, you can leave this as None.
LANGFUSE_DATASET_NAME = None  # e.g. "campaign_relevance_02e1a68ccb0f" or versioned name

# Output CSV (same format as original dataset CSV)
OUT_CSV = ROOT / "input" / "disagreement_subset.csv"

# Choose which IDs to include
ONLY_DISAGREE = True  # True => only posts where models disagree; False => all 300 baseline IDs

# --- load baseline (300 posts x 14 models) -----------------------------------
with BASELINE_PATH.open("r", encoding="utf-8") as f:
    baseline = json.load(f)

if ONLY_DISAGREE:
    subset_ids = [cid for cid, preds in baseline.items()]
else:
    subset_ids = list(baseline.keys())

print(f"Total baseline IDs: {len(baseline)}")
print(f"Subset IDs selected: {len(subset_ids)}")

# --- load full dataset rows (from Langfuse) ----------------------------------
rows = load_dataset_rows(
    csv_path=None,  # force Langfuse path
    langfuse_dataset_name=LANGFUSE_DATASET_NAME,
    body_template_path=None,
)
by_id = {r["custom_id"]: r for r in rows}
print(f"Loaded {len(rows)} rows from dataset; matching on custom_id")

missing = [cid for cid in subset_ids if cid not in by_id]
if missing:
    print(f"WARNING: {len(missing)} IDs from baseline not found in dataset, e.g. {missing[:5]}")

# --- write CSV in the same format as load_dataset.py expects -----------------
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    # Header compatible with load_dataset.load_csv_rows + sync_dataset.py
    writer.writerow(["custom_id", "body", "campaign_relevant", "relevancy_reasoning"])
    for cid in subset_ids:
        row = by_id.get(cid)
        if not row:
            continue
        body = row["body"]          # dict: {store,text,metadata,input,...}
        label = "true" if row["campaign_relevant"] else "false"
        writer.writerow([cid, repr(body), label, ""])

print(f"Wrote subset CSV to: {OUT_CSV}")

Total baseline IDs: 300
Subset IDs selected: 300
Loaded 1199 rows from dataset; matching on custom_id
Wrote subset CSV to: /Users/georgiiburdi/DataspellProjects/benchmark_eval/input/disagreement_subset.csv


In [5]:
!python scripts/sync_dataset.py --csv data/disagreement_subset.csv --name campaign_relevance_disagree_subset

Uploaded 300 items to Langfuse dataset: campaign_relevance_disagree_subset_9d488308aa46
